# ex: getting wordcounts from a book

In [ ]:
from pyspark import SparkContext, SparkConf
from prettytable import PrettyTable

conf = SparkConf().setMaster('local[*]').setAppName('pnp_wordcount')
sc = SparkContext(conf = conf)

book = sc.textFile('./pride_and_prejudice.txt')
words = book.flatMap(lambda x: x.split())
wordCounts = words.countByValue()

table = PrettyTable()
table.field_names = ["Word", "Count"]

for word, count in wordCounts.items():
    table.add_row([word, count])

print(table)
sc.close()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/30 10:39:26 WARN Utils: Your hostname, MerielesPC, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/30 10:39:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/30 10:39:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+----------------------------------+-------+
|               Word               | Count |
+----------------------------------+-------+
|               The                |  285  |
|             Project              |   79  |
|            Gutenberg             |   60  |
|              eBook               |   6   |
|                of                |  3899 |
|              Pride               |   7   |
|               and                |  3444 |
|            Prejudice             |   5   |
|               This               |   49  |
|                is                |  861  |
|               for                |  1041 |
|               the                |  4509 |
|               use                |   23  |
|              anyone              |   20  |
|             anywhere             |   3   |
|                in                |  1923 |
|              United              |   15  |
|              States              |   7   |
|               most               |  196  |
|         

## improved wordcount

In [2]:
from pyspark import SparkContext, SparkConf
from prettytable import PrettyTable
import re

conf = SparkConf().setMaster('local[*]').setAppName('pnp_wordcount')
sc = SparkContext.getOrCreate(conf)

WORD_RE = re.compile(r"\b\w+(?: '\w+)*\b", re.UNICODE)

def normalize_word(line):
    return WORD_RE.findall(line.lower())

book = sc.textFile('./pride_and_prejudice.txt')
words = book.flatMap(normalize_word)
wordCounts = words.map(lambda x: (x, 1)).reduceByKey(lambda x, y: x + y)
wordCountsSorted = wordCounts.sortBy(lambda x: x[1], ascending=False)

table = PrettyTable()
table.field_names = ["Word", "Count"]

for word, count in wordCountsSorted.collect():
    table.add_row([word, count])

print(table)

sc.stop()

+-------------------+-------+
|        Word       | Count |
+-------------------+-------+
|        the        |  4846 |
|         to        |  4405 |
|         of        |  3962 |
|        and        |  3835 |
|        her        |  2260 |
|         i         |  2098 |
|         a         |  2094 |
|         in        |  2051 |
|        was        |  1874 |
|        she        |  1732 |
|        that       |  1620 |
|         it        |  1603 |
|        not        |  1520 |
|        you        |  1417 |
|         he        |  1350 |
|        his        |  1289 |
|         be        |  1280 |
|         as        |  1239 |
|        had        |  1181 |
|        with       |  1149 |
|        for        |  1118 |
|        but        |  1040 |
|         is        |  945  |
|        have       |  883  |
|         at        |  831  |
|         mr        |  807  |
|        him        |  765  |
|         on        |  745  |
|         by        |  724  |
|         my        |  710  |
|         

# challenge: customer totals sorted

In [2]:
from pyspark.sql import SparkSession
from pandas import DataFrame as df
from prettytable import PrettyTable

spark = SparkSession.builder \
    .appName("CustomerTotals") \
    .master("local") \
    .getOrCreate()

df = spark.read.csv(\
    "./customer-orders.csv", 
    schema = "customer_id STRING, item_id STRING, order_price DOUBLE"
)

df.createOrReplaceTempView("orders")

customerTotal = spark.sql("""
                          SELECT customer_id, SUM(order_price) AS total
                          FROM orders
                          GROUP BY customer_id
                          ORDER BY total DESC
                          """)

table = PrettyTable()
table.field_names = ["Customer ID", "Total Order Price"]

for row in customerTotal.collect():
    table.add_row([row.customer_id, row.total])

print(table)
spark.stop()

+-------------+--------------------+
| Customer ID | Total Order Price  |
+-------------+--------------------+
|      68     | 6375.449999999997  |
|      73     | 6206.199999999999  |
|      39     | 6193.109999999999  |
|      54     | 6065.389999999999  |
|      71     | 5995.660000000003  |
|      2      |      5994.59       |
|      97     | 5977.189999999995  |
|      46     | 5963.109999999999  |
|      42     | 5696.840000000003  |
|      59     |      5642.89       |
|      41     |      5637.62       |
|      0      | 5524.949999999998  |
|      8      | 5517.240000000001  |
|      85     |      5503.43       |
|      61     | 5497.479999999998  |
|      32     | 5496.050000000004  |
|      58     | 5437.7300000000005 |
|      63     | 5415.150000000001  |
|      15     | 5413.510000000001  |
|      6      | 5397.879999999998  |
|      92     | 5379.280000000002  |
|      43     |      5368.83       |
|      70     | 5368.249999999999  |
|      72     |      5337.44       |
|